# 01 · Download and inspect the data

All datasets live in one Zenodo record: **[10.5281/zenodo.23153537](https://doi.org/10.5281/zenodo.23153537)**
(three archives: `annotation`, `drug_response`, `gene_perturbation`). Raw GDSC dose-response files are *not* in the record;
download them from the GDSC website if you want to rebuild the response table (optional, see the end).

```
annotation/      raw/ (original .h5ad) · processed/ (labels, Ensembl ids, master_split) · splits/ (cell_id → split)
drug_response/   response_table_merged_leave_drug_out.csv · drug graph + fingerprint features · cellline_expression.h5ad · raw/DepMap_Model.csv
gene_perturbation/  norman.zip · adamson.zip (GEARS PertData) · GEARS simulation split (seed 3) · GO resources
```
The archives are `.rar`: you need **7-Zip** (`7z`), `unrar`, or `pip install rarfile` plus unrar on your PATH.

In [ ]:
from pathlib import Path
from heterosc.zenodo import download_data, ZENODO_DOI

DATA = Path("data")
paths = download_data(DATA)             # ~650 MB, MD5-verified; skips files that are already present
print("DOI:", ZENODO_DOI); paths

## Annotation data

In [ ]:
import pandas as pd, anndata as ad

summary = pd.read_csv(DATA / "annotation" / "annotation_split_summary.csv"); summary

In [ ]:
a = ad.read_h5ad(DATA / "annotation" / "processed" / "Myeloid_ready_master_split.h5ad", backed="r")
print(a.n_obs, "cells x", a.n_vars, "genes")
print(a.obs["master_split"].value_counts().to_dict())          # stratified 70/15/15, seed 42, shared by all four models
print(a.obs["cell_type"].nunique(), "cell types")
a.file.close()

## Drug-response data
The table uses **10-fold leave-drug-out** folds: every drug belongs to exactly one fold, so a test drug is never seen during
training. `assert_leave_drug_out` verifies this and fails loudly otherwise.

In [ ]:
from heterosc.data import load_drug_tables
from heterosc.drug_response import assert_leave_drug_out

resp, graphs, fp = load_drug_tables(DATA / "drug_response")
n_folds = assert_leave_drug_out(resp)
print(f"{len(resp):,} pairs | {resp.drug_id.nunique()} drugs | {resp.cell_line_id.nunique()} cell lines | {n_folds} folds")
resp.groupby("fold").agg(pairs=("drug_id", "size"), drugs=("drug_id", "nunique")).T

## Gene-perturbation data
GEARS reads Norman / Adamson from a folder in its own `PertData` format. Unzip them next to each other:

In [ ]:
import zipfile
PERT = DATA / "gene_perturbation"
for name in ("norman", "adamson"):
    if not (PERT / name).exists():
        with zipfile.ZipFile(PERT / f"{name}.zip") as z:
            z.extractall(PERT)
sorted(p.name for p in PERT.iterdir())

## Optional: raw GDSC tables
Only needed to rebuild the response table from scratch. GDSC data are used under the
[GDSC data usage policy](https://depmap.sanger.ac.uk/documentation/data-usage-policy/); write down the release and date.

In [ ]:
# from heterosc.gdsc import download_gdsc
# download_gdsc(DATA / "gdsc")

## Preparing your own annotation dataset
`scripts/prepare_annotation_data.py` standardises labels, removes cell types with fewer than 10 cells, fills Ensembl ids
(needed by Geneformer) and writes the stratified 70/15/15 master split (seed 42) that **all four models must share**:

```bash
python scripts/prepare_annotation_data.py --input my_data.h5ad --name MyData --label-col cell_type --out data/annotation/processed
```